# LSTM: Long short-term Memory

source: <https://www.tensortonic.com/research?paper=lstm>

Inside an LSTM cell, there are three gates:<br>
1. Forget gate 
2. Input gate (contains two parts: input gate and candidate memory)
3. Output gate (contains two parts: output gate and hidden state)

<img src="https://colah.github.io/posts/2015-08-Understanding-LSTMs/img/LSTM3-chain.png" >

## Forget Gate

<b>f(t)=σ(Wf⋅[ht−1,xt]+bf)</b>
<br>
where: <br>
σ represents the sigmoid function <br>
Wf is the weight matrix for the forget gate <br>
ht−1 is the hidden state from the previous time step <br>
xt is the input at the current time step <br>
bf is the bias for the forget gate <br>


Explanation:<br>
Initial paper didnot consider the forget gate, hence it had a major flaw of not being able to forget the irrelevant memory/information, which kept on accumulating and eventually leading to the problem of vanishing/exploding gradients. Later forget gate was introduced to solve this flaw. <br>

THe value of f(t) is between 0 and 1, where 0 means "completely forget" and 1 means "completely remember". The previous and current information is concatenated and multiplied by the weight matrix and bias is added, this gives us a value which is passed through the sigmoid function to get the final output of the forget gate. This output is then used to decide how much of the previous memory should be retained and how much should be forgotten.

In [6]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def forget_gate(h_prev: np.ndarray, x_t: np.ndarray,
                W_f: np.ndarray, b_f: np.ndarray) -> np.ndarray:

    """Compute forget gate: f_t = sigmoid(W_f @ [h, x] + b_f)"""
    combined_state = np.concatenate((h_prev,x_t),axis=-1) # [h_prev,x_t]
    zf = combined_state @ W_f.T + b_f 
    f_t = sigmoid(zf)
    return f_t

## Input Gate

The input gate determines what new information should be added to the LTM(C) from STM(h), and it has two parts: the input gate(i_t) and the candidate memory(C_tilde). <br> 
<b>i(t)=σ(Wi⋅[ht−1,xt]+bi)</b>
<br>
<b>C_tilde=tanh(WC⋅[ht−1,xt]+bc)</b>
<br>
where: <br>
σ represents the sigmoid function <br>
tanh represents the hyperbolic tangent function <br>
Wi is the weight matrix for the input gate <br>
WC is the weight matrix for the candidate memory <br>
ht−1 is the hidden state from the previous time step <br>
xt is the input at the current time step <br>
bi is the bias for the input gate <br>
bc is the bias for the candidate memory <br>


In [7]:
def input_gate(h_prev: np.ndarray, x_t: np.ndarray,
               W_i: np.ndarray, b_i: np.ndarray,
               W_c: np.ndarray, b_c: np.ndarray) -> tuple:
    """Compute input gate and candidate memory."""

    i_t = sigmoid(np.concatenate((h_prev,x_t),axis=-1) @ W_i.T + b_i)
    c_tilde = np.tanh(np.concatenate((h_prev,x_t),axis=-1) @ W_c.T + b_c)

    return (i_t,c_tilde)

### Cell State Update
After calculating forget gate [f(t)], and input gate [i(t) and C_tilde], we can update the cell state (C) as follows: <br>
<b>C(t)=f(t)∗C(t−1)+i(t)∗C_tilde</b>
<br>

In [8]:
def update_cell_state(C_prev: np.ndarray, f_t: np.ndarray,
                      i_t: np.ndarray, c_tilde: np.ndarray) -> np.ndarray:
    """Update cell state: C_t = f_t * C_prev + i_t * c_tilde"""

    C_t = f_t * C_prev + i_t * c_tilde
    return C_t

## Output Gate
The output gate determines what information from the LTM ( C) should be outputted as the hidden state STM (h).
<br>
It also has two parts: <br>
1. Output gate (o_t)
2. Hidden state (h_t)
<br>
<b>o(t)=σ(Wo⋅[ht−1,xt]+bo)</b>
<br>
<b>h(t)=o(t)∗tanh(C(t))</b>
<br>
where: <br>
σ represents the sigmoid function <br>
tanh represents the hyperbolic tangent function <br>
Wo is the weight matrix for the output gate <br>
ht−1 is the hidden state from the previous time step <br>
xt is the input at the current time step <br>
bo is the bias for the output gate <br>
C(t) is the long-term memory at the current time step <br>

In [9]:
def output_gate(h_prev: np.ndarray, x_t: np.ndarray, C_t: np.ndarray,
                W_o: np.ndarray, b_o: np.ndarray) -> tuple:
    """Compute output gate and hidden state."""

    o_t = sigmoid(np.concatenate((h_prev,x_t),axis=-1) @ W_o.T + b_o)

    h_t = np.tanh(C_t) * o_t

    return (o_t,h_t)